In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../../"))

import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler



In [ ]:
df_cancer = pd.read_csv('../../../data/raw/structured/breastCancer.csv')
print("Data loaded successfully!")
df_cancer.head()

In [ ]:
# Data preprocessing
df_cancer.drop(["id"], axis=1, inplace=True)

y = df_cancer["diagnosis"].values
X = df_cancer.drop(["diagnosis"], axis=1)

# Scale features
scaler = StandardScaler()
scaler.fit(X)
X = scaler.transform(X)
X = pd.DataFrame(X, columns=df_cancer.columns[1:])

print("Preprocessing complete!")
X.head()

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# Train model
clf = KNeighborsClassifier()
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluate
print(f"Accuracy: {clf.score(X_test, y_test):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Cross-validation
kf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

all_y_true = []
all_y_pred = []
accuracy_scores = []

for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    clf = KNeighborsClassifier(n_neighbors=3)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    accuracy_scores.append(accuracy_score(y_test, y_pred))
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

print(f"Mean Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Std Accuracy: {np.std(accuracy_scores):.4f}")
print("\nOverall Classification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=['Benign', 'Malignant']))